# ColumnTransformer

`ColumnTransformer` permite **aplicar diferentes transformaciones a diferentes columnas dentro del mismo conjunto de datos**.

Esto resuelve varios problemas comunes:

1. Mezcla de variables numéricas y categóricas

En un dataset típico es normal tener columnas como estas:

*   Edad (numérica) → requiere escalado.
*   Ingreso (numérica) → requiere normalización.
*   Ciudad (categórica) → requiere *one‑hot encoding*.
*   Estado civil (categórica) → puede requerir ordinal encoding.

Sin `ColumnTransformer`, habría que separar manualmente columnas, transformar cada subconjunto y luego volver a unirlos. Esto es tedioso y propenso a errores.

2. Código más limpio y fácil de mantener

En lugar de escribir muchas líneas de código para manejar columnas por separado, `ColumnTransformer` permite declarar todas las operaciones en un solo bloque, documentadas y ordenadas.

3. Evita errores al mezclar transformaciones

Si por error se intenta aplicar un escalador numérico a una columna categórica o viceversa, aparecerían errores. `ColumnTransformer` evita esto porque define explícitamente qué transformación va en cada conjunto de columnas.

4. Preprocesamiento reproducible

Deja registrado *qué se aplicó*, *a qué columnas* y *en qué orden*, lo cual es crucial cuando el proyecto crece o se integra con otros equipos.

## Ejemplo de uso

Se va a entrenar un modelo con el dataset **adult.data**, que tiene como objetivo predecir la variable **income**.

Sa va a hacer el siguiente preprocesamiento a las variables del dataset:

* **age** y **hours-per-week** se van a estandarizar con `StandardScaler`.
* **fnlwgt** se va a transformar con `PowerTransformer`.
* **education** se va a codificar con `OrdinalEncoder`.
* **workclass**, **marital-status**, **occupation**, **relationship**, **race** y **sex** se van a codficar con `OneHotEncoder`.
* Las otras variables se van a descartar del modelo.

Empezamos cargando el dataset, y por facilidad, eliminando los datos nulos.

In [1]:
import pandas as pd

df = pd.read_csv(
    "http://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    header =None,
    names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
             'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
             'hours-per-week', 'native-country', 'income'],
    na_values= [' ?']
    )

df.dropna(inplace=True)

df.sample(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
22922,60,State-gov,198815,Assoc-acdm,12,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,20,Mexico,<=50K
22129,49,Local-gov,329144,Some-college,10,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,44,United-States,>50K
23242,20,Private,286166,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,48,United-States,<=50K
28363,52,Self-emp-inc,68015,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,90,United-States,>50K
6486,57,Local-gov,215175,Masters,14,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,60,United-States,<=50K


A continuación, partimos los datos en subconjuntos de entrenamiento y prueba.

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(['income'], axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.7)
print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (21113, 14)
Tamaño del conjunto de prueba es: (9049, 14)


Ahora procedemos a preprocesar los subconjuntos por separado.

Primero se **declaran** los transformadores que se van a usar:

In [5]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer

ss = StandardScaler() # Para preprocesar age y hours-per-week
pt = PowerTransformer() # Para preprocesar fnlwgt
orden = [' Preschool', ' 1st-4th', ' 5th-6th', ' 7th-8th', ' 9th', ' 10th', ' 11th', ' 12th',
         ' HS-grad',' Some-college', ' Assoc-acdm', ' Assoc-voc', ' Bachelors', ' Masters',
         ' Prof-school', ' Doctorate']
ore = OrdinalEncoder(categories=[orden], dtype='int') # Para preprocesar education
ohe = OneHotEncoder(sparse_output=False, drop='if_binary') # workclass, marital-status, occupation, relationship, race y sex

En segundo lugar, con `ColumnTransformer` se crea el preprocesador en el que se indica a que variables se les va a aplicar cada transformación:

In [6]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('num_prep', ss, ['age', 'hours-per-week']), # Escalado
    ('num_prep2', pt, ['fnlwgt']), # Transformación
    ('cod_education', ore, ['education']), # Codificación ordinal
    ('cod_oh', ohe, ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex'])
    ], remainder='drop') # Las demás columnas se descartan

In [9]:
preprocessor

ColumnTransformer(transformers=[('num_prep', StandardScaler(),
                                 ['age', 'hours-per-week']),
                                ('num_prep2', PowerTransformer(), ['fnlwgt']),
                                ('cod_education',
                                 OrdinalEncoder(categories=[[' Preschool',
                                                             ' 1st-4th',
                                                             ' 5th-6th',
                                                             ' 7th-8th', ' 9th',
                                                             ' 10th', ' 11th',
                                                             ' 12th',
                                                             ' HS-grad',
                                                             ' Some-college',
                                                             ' Assoc-acdm',
                                                             ' Assoc-voc',
                                                             ' Bachelors',
                                                             ' Masters',
                                                             ' Prof-school',
                                                             ' Doctorate']],
                                                dtype='int'),
                                 ['education']),
                                ('cod_oh',
                                 OneHotEncoder(drop='if_binary',
                                               sparse_output=False),
                                 ['workclass', 'marital-status', 'occupation',
                                  'relationship', 'race', 'sex'])])

Ahora si se aplican los preprocesamientos a los subconjuntos de entrenamiento y prueba:

In [7]:
preprocessor.fit(X_train)
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

Y ahora si, puedo entrenar mi modelo:

In [8]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=1
    )

tree.fit(X_train_prep, y_train)

DecisionTreeClassifier(max_depth=5, random_state=1)

Ya puedo evaluar el modelo entrenado.

In [12]:
print(f'Accuracy en el conjunto de entrenamiento: {tree.score(X_train_prep, y_train)}')
print(f'Accuracy en el conjunto de prueba: {tree.score(X_test_prep, y_test)}')

Accuracy en el conjunto de entrenamiento: 0.8189267276085824
Accuracy en el conjunto de prueba: 0.8197590894021439


Si se desea saber cuales fueron las características con las que se entrenó el modelo, se puede usar el método `get_features_names_out()`:

In [15]:
print(preprocessor.get_feature_names_out())

['num_prep__age' 'num_prep__hours-per-week' 'num_prep2__fnlwgt'
 'cod_education__education' 'cod_oh__workclass_ Federal-gov'
 'cod_oh__workclass_ Local-gov' 'cod_oh__workclass_ Private'
 'cod_oh__workclass_ Self-emp-inc' 'cod_oh__workclass_ Self-emp-not-inc'
 'cod_oh__workclass_ State-gov' 'cod_oh__workclass_ Without-pay'
 'cod_oh__marital-status_ Divorced'
 'cod_oh__marital-status_ Married-AF-spouse'
 'cod_oh__marital-status_ Married-civ-spouse'
 'cod_oh__marital-status_ Married-spouse-absent'
 'cod_oh__marital-status_ Never-married'
 'cod_oh__marital-status_ Separated' 'cod_oh__marital-status_ Widowed'
 'cod_oh__occupation_ Adm-clerical' 'cod_oh__occupation_ Armed-Forces'
 'cod_oh__occupation_ Craft-repair' 'cod_oh__occupation_ Exec-managerial'
 'cod_oh__occupation_ Farming-fishing'
 'cod_oh__occupation_ Handlers-cleaners'
 'cod_oh__occupation_ Machine-op-inspct'
 'cod_oh__occupation_ Other-service' 'cod_oh__occupation_ Priv-house-serv'
 'cod_oh__occupation_ Prof-specialty'
 'cod_oh_

In [18]:
print('Número de características con las que fue entrenado el modelo: ', preprocessor.get_feature_names_out().shape[0])

Número de características con las que fue entrenado el modelo:  44
